# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name + ': ' + metadata.description)
print(f"Dataset version: {metadata.version}, Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields (variables), and their `@id`s.

We list all record sets, and for each, print their `@id`, `name`, and associated fields (by `@id`).

In [ ]:
# List all available RecordSets in the dataset by their @id
# Also display the fields (columns) for each RecordSet

record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets defined in the schema.")
else:
    for rs in record_sets:
        # Each rs is a RecordSet object
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '<unnamed>')}")
        print(f"  Description: {rs.get('description', '<no description>')}")
        # List fields (columns)
        columns = rs.get('column', [])
        print("  Columns:")
        for col in columns:
            print(f"    Column @id: {col['@id']} Name: {col.get('name', '<unnamed>')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

For this dataset, let's extract all the record sets found above and load their records using the `@id`.

**Note:** If the Croissant schema does not list any record sets directly, you may need to inspect the dataset and infer likely ids, such as those matching the main survey data.

In [ ]:
# Collect the @id values of all record sets
record_sets_ids = []
if dataset.metadata.recordSet:
    for rs in dataset.metadata.recordSet:
        record_sets_ids.append(rs['@id'])
else:
    # Fallback: use a likely record set id manually (example)
    record_sets_ids.append('http://mlcommons.org/croissant/mental_health_survey_records') # replace as appropriate

# Load records for each available record set
dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric mental health assessment score (e.g., `GAD-7` or `PHQ-9`), filter for high scores, normalize, and group by a demographic field (e.g., `level_of_education`).

**All field references use their `@id`.**

In [ ]:
# Example (replace the @id with those found above for your record set)

# Choose the first available record set for demo
if record_sets_ids:
    selected_record_set_id = record_sets_ids[0]
    df = dataframes.get(selected_record_set_id, pd.DataFrame())

    # Identify likely numeric assessment fields by @id (example values, replace as needed)
    numeric_field_id = 'http://mlcommons.org/croissant/GAD7_score'  # Example: GAD-7 score column @id
    group_field_id = 'http://mlcommons.org/croissant/level_of_education'  # Example: level of education column @id

    # For demo, fall back to column names if @id not present
    if numeric_field_id not in df.columns:
        # Try 'GAD7_score' as column name
        numeric_field_id = 'GAD7_score' if 'GAD7_score' in df.columns else df.columns[0]
    if group_field_id not in df.columns:
        group_field_id = 'level_of_education' if 'level_of_education' in df.columns else df.columns[1]

    # Filter records where GAD-7 score exceeds a threshold
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Group records by education level (if present)
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Average {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} not found in dataframe columns.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

Let's plot the histogram of the GAD-7 scores and visualize average GAD-7 score by education level (using the chosen field `@id`s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets_ids:
    selected_record_set_id = record_sets_ids[0]
    df = dataframes.get(selected_record_set_id, pd.DataFrame())
    numeric_field_id = 'http://mlcommons.org/croissant/GAD7_score'
    group_field_id = 'http://mlcommons.org/croissant/level_of_education'
    if numeric_field_id not in df.columns:
        numeric_field_id = 'GAD7_score' if 'GAD7_score' in df.columns else df.columns[0]
    if group_field_id not in df.columns:
        group_field_id = 'level_of_education' if 'level_of_education' in df.columns else df.columns[1]
    
    # Plot histogram of scores
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'{numeric_field_id} Distribution')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Bar plot: average score by education
    if group_field_id in df.columns:
        mean_score_by_group = df.groupby(group_field_id)[numeric_field_id].mean()
        mean_score_by_group.plot(kind='bar', figsize=(8,4))
        plt.title(f'Average {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR² mental health dataset's metadata and records using `mlcroissant`.
- Explored record sets, fields, and their `@id`s for semantic clarity.
- Extracted data and performed preliminary EDA, including filtering and normalization of mental health scores.
- Visualized assessment score distributions and their relation to education levels.

This demonstrates a reproducible, standards-driven workflow for dataset exploration and analysis using Croissant schemas and the `mlcroissant` Python library.